In [0]:
#Create a volume to store your checkpoints.  Databricks Free Edition has completely disabled access to the public DBFS root
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.checkpoints")

In [0]:
# Rate source generates rows continuously (10 per second)
rate_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)  # slow down to save compute
    .load()
)
# display(rate_df, checkpointLocation = "/Volumes/workspace/default/checkpoints/rate_stream")   # This STARTS the stream and shows live output

# What you'll see: A table that updates every few seconds with new rows. 

# Stop it — click the "Cancel" button above the display.

In [0]:
# # Read the Streaming Table as a stream
# df = (spark.readStream
#       .format("delta")
#       .table("workspace.default.silver_clickstream") # Or .load("/path/to/table")
#      )

# # Write it somewhere else (e.g., another table)
# (df.writeStream
#   .format("delta")
#   .option("checkpointLocation", "/checkpoints/my_checkpoint")
#   .toTable("workspace.default.some_other_table")
# )

## read from kafka
#df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "...").load()

## Fixing the Checkpoint Mismatch Error

Your previous attempt failed with a **DBFS error**. When you fixed the path and tried again, the code you ran was slightly different from what the system expected, based on the earlier failed attempt.

Databricks keeps strict records of **stateful operators** in the checkpoint metadata. Any mismatch blocks the new run to prevent data corruption.

**Fix:** remove the leftover checkpoint files from the earlier attempt so the stream can start fresh. Run this in a new cell:

```python
dbutils.fs.rm("/tmp/checkpoints/...", True)
```

In [0]:
# dbutils.fs.rm("/Volumes/workspace/default/checkpoints/rate_stream", True)

In [0]:
# Write the stream to a Delta table
(rate_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/rate_stream") 
    .trigger(availableNow=True)
    .toTable("workspace.default.rate_output")
)

**Why Checkpoints are Mandatory (The "Brain" of the Stream)**
```
A checkpoint location is the memory of your Structured Streaming job. It stores two critical things:

Progress: Which files/records have already been processed.

State: For stateful operations (like aggregations or joins), it stores the intermediate data.

What happens if you don't provide a checkpoint:

Every time the job runs, it starts from scratch. 
It forgets what it processed yesterday. If you are reading from S3, 
it will re-read every single file from the beginning of time, leading to massive duplication in your target table.
```

In [0]:
%sql
SELECT * FROM workspace.default.rate_output ORDER BY timestamp DESC LIMIT 20;

`You just did Structured Streaming end-to-end.`

#Stream-Static Join
`We'll join the rate stream with a small static table.`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.static_lookup AS
SELECT * FROM VALUES
  (0, 'Zero'),
  (1, 'One'),
  (2, 'Two'),
  (3, 'Three'),
  (4, 'Four')
AS t(id, name);

In [0]:
from pyspark.sql.functions import col

# Stream (rate source)
rate_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("id", (col("value") % 5).cast("int"))    # map to 0-4
)

# Static
static_df = spark.read.table("workspace.default.static_lookup")

# Join
joined_df = rate_df.join(static_df, "id", "inner")

# Write
(joined_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/join_stream")
    .trigger(availableNow=True)
    .toTable("workspace.default.joined_output")
)

In [0]:
%sql
SELECT * FROM workspace.default.joined_output ORDER BY timestamp DESC LIMIT 20;

#foreachBatch with MERGE (Upsert)
`This simulates real-world CDC-like behavior using plain Structured Streaming.`

In [0]:
%sql
--Setup target table
CREATE OR REPLACE TABLE workspace.default.user_state (
  user_id INT,
  balance INT
);

INSERT INTO workspace.default.user_state VALUES (1, 100), (2, 200);

In [0]:
#Streaming upsert with foreachBatch
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Stream of "changes"
changes_df = (spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    .withColumn("user_id", (col("value") % 2 + 1).cast("int"))   # users 1 and 2
    .withColumn("balance", (col("value") * 10).cast("int"))
    .select("user_id", "balance")
)

# foreachBatch function: MERGE each micro-batch
def upsert_to_delta(microBatchDF, batchId):
    target = DeltaTable.forName(spark, "workspace.default.user_state")
    (target.alias("t")
        .merge(microBatchDF.alias("s"), "t.user_id = s.user_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# Run the streaming job
(changes_df.writeStream
    .foreachBatch(upsert_to_delta)
    .outputMode("update")
    .option("checkpointLocation", "/Volumes/workspace/default/checkpoints/upsert_stream")
    .trigger(availableNow=True) #The stream will not run forever. It processes what's there, then exits cleanly.
    .start()
)

## Compare the Triggers

| Trigger | Behavior |
|---|---|
| *(default: nothing specified)* | Runs forever, processing as fast as possible |
| `processingTime="10 seconds"` | Runs forever, every 10 seconds |
| `availableNow=True` | Processes new data, then **STOPS** ✅ |
| `once=True` *(deprecated)* | Same as `availableNow` |

## Free Edition Tips

1. Always set `rowsPerSecond` low (1–5) to reduce compute.
2. Always use `trigger(availableNow=True)` so the stream stops after processing instead of running forever.
3. Delete old checkpoint folders when re-running experiments:

   ```python
   dbutils.fs.rm("/tmp/checkpoints/...", True)
   ```
````

In [0]:
%sql
SELECT * FROM workspace.default.user_state ORDER BY user_id;

In [0]:
%sql
--when done to free up space in free edition
DROP TABLE IF EXISTS workspace.default.rate_output;
DROP TABLE IF EXISTS workspace.default.joined_output;
DROP TABLE IF EXISTS workspace.default.user_state;
DROP TABLE IF EXISTS workspace.default.static_lookup;